# TAC calibration — 01 source extraction

Writes `data/sources_register.csv`. Run this before `02_tac_calibration.ipynb`:
02 checks every source it cites against the register this notebook produces.

The register is the single source of truth for TAC provenance — per-country
charging documents, the FX reference, and the evidence behind the escalation
rate. Add a source here, never in the CSV.

In [ ]:
# TAC calibration — source extraction
#
# Single source of truth for the TAC source register. Every row is written
# here; `data/sources_register.csv` is a generated artifact and must never
# be hand-edited (re-run this notebook instead). Same contract as the
# compositions calibration: notebook is truth, CSV is output.
#
# Stored documents follow `{source_id}.{ext}` (roadmap §0.3.3) — the infra
# source ids already encode country, document and year, so the filename is
# derivable from the register and needs no column of its own.

import csv
from pathlib import Path


def _resolve_data_dir() -> Path:
    """Notebook may run from calib/ or from the repo root; resolve either."""
    here = Path.cwd()
    for cand in (here / "data", here / "backend/models/infrastructure/tac/calib/data"):
        if cand.parent.exists():
            cand.mkdir(exist_ok=True)
            return cand
    raise RuntimeError(f"cannot locate the calib data directory from {here}")


DATA_DIR = _resolve_data_dir()
print(f"data directory: {DATA_DIR}")

# Date the register was last reviewed end to end. Held in one place so a
# review pass cannot leave rows carrying stale or drifted access dates.
REGISTER_REVIEWED = "2026-08-11"

REGISTER_COLUMNS = [
    "source_id",
    "short_id",
    "used",
    "downloaded",
    "title",
    "publisher",
    "pub_year",
    "price_basis_year",
    "currency",
    "kind",
    "url_or_file",
    "date_accessed",
    "reliability_note",
]

register_rows: list[tuple] = []

## Per-country charging documents

One row per document a calibrated value leans on. `price_basis_year` is the year the
rates apply to, which is often not the publication year — the distinction drives
the escalation in notebook 02.

In [ ]:
# --- Per-country charging documents (S1) ---
register_rows += [
    (
        "AT-SNNB-2027",
        "at_snnb_2027",
        "Used",
        "x",
        "Schienennetz-Nutzungsbedingungen 2027",
        "ÖBB-Infrastruktur AG",
        2025,
        2027,
        "EUR",
        "network_statement",
        "https://infrastruktur.oebb.at/de/geschaeftspartner/schienennetz/snnb/snnb-2027/schienennetz-nutzungsbedingungen-2027.pdf",
        REGISTER_REVIEWED,
        "Train-km and gross-tonne-km rates for Personenfernverkehr (Tab.11/13, ch.5.3.4); congestion surcharge §5.4. IRG survey 2025 gives 0.649 / 0.002129 — consistent, small yearly drift",
    ),
    (
        "AT-SNNB-2026",
        "at_snnb_2026",
        "Used",
        "x",
        "Schienennetz-Nutzungsbedingungen 2026",
        "ÖBB-Infrastruktur AG",
        2024,
        2026,
        "EUR",
        "network_statement",
        "https://infrastruktur.oebb.at/de/geschaeftspartner/schienennetz/snnb",
        REGISTER_REVIEWED,
        "Not used for TAC (superseded by the 2027 edition); retained because the electricity, shunting and parking tables of the 2026 edition feed the other infrastructure domains",
    ),
    (
        "BE-NS-2027",
        "be_ns_2027",
        "Used",
        "x",
        "Network Statement 2027 (version 30 June 2026)",
        "Infrabel",
        2026,
        2026,
        "EUR",
        "network_statement",
        "https://infrabel.be/en/networkstatement",
        REGISTER_REVIEWED,
        "Direct cost per §5.3 plus Appendix F.2 workbook (sheets 2.1.1, 2.1.2.3) for the Ramsey-Boiteux mark-up matrix. IRG survey 2025 DC 2.076386 matches the indexation chain 2023 to 2026",
    ),
    (
        "BG-NRIC-2026",
        "bg_nric_2026",
        "Used",
        "x",
        "Charges and Prices, Annex 5.3.2 v.06",
        "NRIC (National Railway Infrastructure Company)",
        2026,
        2026,
        "EUR",
        "network_statement",
        "https://www.rail-infra.bg/en/353",
        REGISTER_REVIEWED,
        "Section I passenger rates, effective 1 Feb 2026. Power-supply-equipment charge excluded as energy; its unit is ambiguous in the document",
    ),
    (
        "CH-NZV",
        "ch_nzv",
        "Used",
        "x",
        "SR 742.122 Eisenbahn-Netzzugangsverordnung (NZV)",
        "Swiss Confederation (Fedlex)",
        2026,
        2026,
        "CHF",
        "regulation",
        "https://www.fedlex.admin.ch/eli/cc/1999/142/de#a21",
        REGISTER_REVIEWED,
        "Art.19a/20/20a: Haltezuschlag, Deckungsbeitrag, long-train rebate. Legal text rather than a network statement — CH prices are set by ordinance",
    ),
    (
        "CH-NZV-BAV",
        "ch_nzv_bav",
        "Used",
        "x",
        "SR 742.122.4 NZV-BAV (BAV ordinance on network access)",
        "Federal Office of Transport (BAV)",
        2026,
        2026,
        "CHF",
        "regulation",
        "https://www.fedlex.admin.ch/eli/cc/2012/371/de",
        REGISTER_REVIEWED,
        "Art.1 and Anhänge: line-category base prices, wear price, peak factor. Status 1 Feb 2026",
    ),
    (
        "CZ-NS-2027",
        "cz_ns_2027",
        "Used",
        "x",
        "Network Statement 2027 (EN web version)",
        "Správa železnic",
        2026,
        2027,
        "CZK",
        "network_statement",
        "https://www.spravazeleznic.cz/web/en/network-statement-2027",
        REGISTER_REVIEWED,
        "Charging annex: ZI gross-tonne-km rate for 1 Jan to 11 Dec 2027. IRG survey composite 30.34 CZK/trkm is consistent with the ZI x M structure",
    ),
    (
        "DE-INB-2026",
        "de_inb_2026",
        "Used",
        "x",
        "INB 2026 Anlage 5.3 (Redaktionsstand 12 Dec 2025)",
        "DB InfraGO AG",
        2025,
        2026,
        "EUR",
        "network_statement",
        "https://www.dbinfrago.com/web/schienennetz/leistungen/trassen/trassenpreissystem-2026/schienenpersonenfernverkehr-spfv",
        REGISTER_REVIEWED,
        "SPFV market segment Nacht, valid from 14 Dec 2025; segment definition Ziffer 5.3.2.5. Prints 3.33 EUR/trkm before the BNetzA re-approval adjustment",
    ),
    (
        "DE-BNETZA-BK10",
        "de_bnetza_bk10",
        "Used",
        "x",
        "BK10-25-0067 Anlage 2 — approved consolidated INB 2026",
        "Bundesnetzagentur",
        2025,
        2026,
        "EUR",
        "regulatory_decision",
        "https://www.bundesnetzagentur.de/DE/Beschlusskammern/1_GZ/BK10-GZ/2025/2025_0001bis0099/BK10-25-0067/Anlagen/BK10-25-0067_Z_Anlage2_Download.pdf?__blob=publicationFile&v=2",
        REGISTER_REVIEWED,
        "The approval decision behind the INB rates. Does NOT contain the -17% SPFV re-approval of 22 Jul 2026 that the DE value assumes — that document is still to be filed (open action 1)",
    ),
    (
        "DK-BEK-2024",
        "dk_bek_2024",
        "Used",
        "-",
        "Bekendtgørelse om infrastrukturafgifter m.v. (BEK 2024/1351)",
        "Danish Transport Ministry / Banedanmark",
        2024,
        2025,
        "DKK",
        "regulation",
        "MISSING",
        REGISTER_REVIEWED,
        "Rates captured via the IRG-Rail TAC survey; the executive order itself is NOT on disk. NS 2027 carries no numbers and refers to the order, which is indexed annually. IRG survey 0.78 EUR/trkm matches 5.80 DKK",
    ),
    (
        "EE-TTJA-2026",
        "ee_ttja_2026",
        "Used",
        "x",
        "Raudteeinfrastruktuuri kasutustasu määrad (published rate table)",
        "Tarbijakaitse ja Tehnilise Järelevalve Amet (TTJA)",
        2026,
        2026,
        "EUR",
        "tariff_decision",
        "https://ttja.ee/ariklient/raudtee/kasutustasu-maarad",
        REGISTER_REVIEWED,
        "TT 2025/26 base rates. Mark-up for domestic passenger service is positive but zero for international — night trains take the base only. IRG survey EE sheet is empty, so this page is the better source (one-sided cross-check)",
    ),
    (
        "ES-BOE-2024",
        "es_boe_2024",
        "Used",
        "x",
        "BOE-A-2024-22140 (consolidated) — railway charges",
        "Boletín Oficial del Estado",
        2024,
        2023,
        "EUR",
        "regulation",
        "https://www.boe.es/buscar/act.php?id=BOE-A-2024-22140",
        REGISTER_REVIEWED,
        "Art.4 modality rates by line type, Art.5 the Madrid-Barcelona-Frontera seat-km surcharge. Rates in force since 2023",
    ),
    (
        "ES-ADIF-2027",
        "es_adif_2027",
        "Used",
        "x",
        "Declaración sobre la Red 2027 (NS ADIF V1)",
        "Adif / Adif Alta Velocidad",
        2026,
        2027,
        "EUR",
        "network_statement",
        "https://www.adif.es/sobre-adif/declaracion-red",
        REGISTER_REVIEWED,
        "Network statement context for the ES charges; the rates themselves are set in the BOE regulation",
    ),
    (
        "FI-NS-2027",
        "fi_ns_2027",
        "Used",
        "x",
        "Verkkoselostus / Network Statement 2027",
        "Väylävirasto (Finnish Transport Infrastructure Agency)",
        2026,
        2027,
        "EUR",
        "network_statement",
        "https://www.doria.fi/bitstream/handle/10024/195216/vj_2026-38eng_978-952-405-425-6.pdf?sequence=1&isAllowed=y",
        REGISTER_REVIEWED,
        "Tab.2 §5.3 basic charge per gross-tonne-km. FI charges no train-km term — the NULL b_day is a documented tariff fact, not missing data",
    ),
    (
        "FR-DRR-2027-A52",
        "fr_drr_2027_a52",
        "Used",
        "x",
        "DRR 2027 Appendix 5.2 — scale of minimum services",
        "SNCF Réseau",
        2026,
        2027,
        "EUR",
        "network_statement",
        "https://www.sncf-reseau.com/en/drr/network-statement-national-rail-network-timetable-2027",
        REGISTER_REVIEWED,
        "App 5.2.2, UIC line groups 2-6. The gross-tonne-km term is published per 1000 CGT-km. The night-train column shows '-' in the RM table",
    ),
    (
        "FR-DRR-A512",
        "fr_drr_a512",
        "Used",
        "x",
        "DRR 2027 Annexe 5.1.2",
        "SNCF Réseau",
        2026,
        2027,
        "EUR",
        "network_statement",
        "https://www.sncf-reseau.com/en/drr/network-statement-national-rail-network-timetable-2027",
        REGISTER_REVIEWED,
        "Supporting annex to the FR charge structure",
    ),
    (
        "GR-OSE-2026",
        "gr_ose_2026",
        "Used",
        "x",
        "Network Statement 2026 (EN final)",
        "OSE",
        2026,
        2019,
        "EUR",
        "network_statement",
        "https://ose.gr/wp-content/uploads/2026/02/OSE_2026-ENG_Final.pdf",
        REGISTER_REVIEWED,
        "Ch.6 charge formula. Rates are a 2019 base that the statement itself inflates (x1.1975) and applies a 0.60 recovery factor to — hence the 2019 price basis and the longest escalation path to 2032",
    ),
    (
        "HR-NS-2027",
        "hr_ns_2027",
        "Used",
        "x",
        "Izvješće o mreži / Network Statement 2027",
        "HŽ Infrastruktura",
        2026,
        2027,
        "EUR",
        "network_statement",
        "https://eng.hzinfra.hr/?page_id=284",
        REGISTER_REVIEWED,
        "§5.3: EuroNight train type T 2.10 x line category L1 1.90 x Cvlkm 0.54. L1 assumed for international corridors",
    ),
    (
        "HU-NS-2627",
        "hu_ns_2627",
        "Used",
        "x",
        "Network Statement 2026-2027, Annex 5.2-6",
        "VPE / MÁV",
        2026,
        2027,
        "HUF",
        "network_statement",
        "https://vpe.kti.hu/en/network-statement/network-statement-2026-2027/",
        REGISTER_REVIEWED,
        "Track category I passenger rate plus path-ensuring component; gross-tonne-km term separate. GYSEV is not calibrated separately (minor route share). Catenary use excluded as energy",
    ),
    (
        "IE-NS-2027",
        "ie_ns_2027",
        "Used",
        "x",
        "Network Statement 2027",
        "Iarnród Éireann",
        2026,
        2027,
        "EUR",
        "network_statement",
        "https://www.irishrail.ie/en-ie/about-us/iarnrod-eireann-network-statement",
        REGISTER_REVIEWED,
        "Ch.6.2/6.3 variable charge. The fixed track access charge is franchise-only and does not apply to an open-access operator",
    ),
    (
        "IT-LISTINO",
        "it_listino",
        "Used",
        "x",
        "Listino Tariffario Pacchetto Minimo di Accesso (PMdA)",
        "RFI",
        2026,
        2027,
        "EUR",
        "tariff_list",
        "https://www.rfi.it/en/railway-infrastructure-access-/Network-statement.html",
        REGISTER_REVIEWED,
        "Component A flat and speed-class terms, Component B Basic FOND Standard Notturno. The Notturno band hours are assumed 22:00-06:00 — not stated in the Listino (open action 3)",
    ),
    (
        "IT-NS-2027",
        "it_ns_2027",
        "Used",
        "x",
        "Network Statement 2027 (June 2026)",
        "RFI",
        2026,
        2027,
        "EUR",
        "network_statement",
        "https://www.rfi.it/en/railway-infrastructure-access-/Network-statement.html",
        REGISTER_REVIEWED,
        "Structural context for the Listino tariff terms",
    ),
    (
        "LT-LTG-2627",
        "lt_ltg_2627",
        "Used",
        "x",
        "Network Statement 2026-2027 and annexes v1",
        "LTG Infra",
        2026,
        2027,
        "EUR",
        "network_statement",
        "https://ltginfra.lt/en/railway-infrastructure/map/network-statements/",
        REGISTER_REVIEWED,
        "Tariff decision §5.3.2 gross-tonne-km rate",
    ),
    (
        "LU-NS-2027",
        "lu_ns_2027",
        "Used",
        "x",
        "Document de référence du réseau 2027 (EN v1.0)",
        "ACF / CFL",
        2026,
        2026,
        "EUR",
        "network_statement",
        "https://acf.gouvernement.lu/en/sillon/Document-de-reference-du-reseau.html",
        REGISTER_REVIEWED,
        "§5.3.2: base cC with the alpha (train length) and beta factors, plus the path-administration term on a regular timetable path",
    ),
    (
        "LV-NS-2027",
        "lv_ns_2027",
        "Used",
        "x",
        "Network Statement 2027",
        "LDz / LatRailNet",
        2026,
        2026,
        "EUR",
        "network_statement",
        "https://www.ldz.lv/en/network-statement-2027",
        REGISTER_REVIEWED,
        "§5.2 international passenger rates within the EEA",
    ),
    (
        "NL-NS-2027",
        "nl_ns_2027",
        "Used",
        "x",
        "Network Statement 2027 (version 1.1, 6 May 2026)",
        "ProRail",
        2026,
        2027,
        "EUR",
        "network_statement",
        "https://www.prorail.nl/samenwerken/vervoerders/network-statement",
        REGISTER_REVIEWED,
        "Train path service, weight class 601-3200 t; the class is resolved from the composition at runtime. All mark-ups are zero for passenger",
    ),
    (
        "NO-NS-2027",
        "no_ns_2027",
        "Used",
        "x",
        "Network Statement 2027 (EN v1.1)",
        "Bane NOR",
        2026,
        2026,
        "NOK",
        "network_statement",
        "https://oppslagsverk.banenor.no/en/network-statement/",
        REGISTER_REVIEWED,
        "Tab.4 §5.3.3. The non-Oslo rate is applied uniformly, which is the conservative choice",
    ),
    (
        "PL-PLK-A91",
        "pl_plk_a91",
        "Used",
        "x",
        "Network Statement 2026/2027 Annex 9.1 (SMK)",
        "PKP Polskie Linie Kolejowe",
        2026,
        2027,
        "PLN",
        "network_statement",
        "https://en.plk-sa.pl/for-customers-and-partners/the-rules-for-allocating-train-paths/network-statement-2026/2027",
        REGISTER_REVIEWED,
        "Base rate multiplied by WM (mass) and WK (line category), both 1.0 for the reference train",
    ),
    (
        "PT-IP-2027",
        "pt_ip_2027",
        "Used",
        "x",
        "1st Addenda to Network Statement 2027",
        "Infraestruturas de Portugal",
        2026,
        2027,
        "EUR",
        "network_statement",
        "https://servicos.infraestruturasdeportugal.pt/sites/default/files/1st%20Addenda%20Network%20Statement%202027_0.pdf",
        REGISTER_REVIEWED,
        "§5.3 category A electric traction, Regular/Peak and Low schedule bands",
    ),
    (
        "RO-CFR-A25",
        "ro_cfr_a25",
        "Used",
        "x",
        "Network Statement Annex 25.a",
        "CFR SA",
        2025,
        2024,
        "RON",
        "network_statement",
        "https://cfr.ro/download-drr-2026-network-statement/",
        REGISTER_REVIEWED,
        "Tc plus Ttsn weight formula at 600 t, train class A. Reproduces the published worked example",
    ),
    (
        "SE-NS-2027",
        "se_ns_2027",
        "Used",
        "x",
        "Network Statement 2027, Annex 1B",
        "Trafikverket",
        2026,
        2027,
        "SEK",
        "network_statement",
        "https://bransch.trafikverket.se/en/startpage/operations/Operations-railway/Network-Statement/network-statement-2027/",
        REGISTER_REVIEWED,
        "Passenger train-km and gross-tonne-km rates, mean axle load <= 17 t. Also the source for the Øresund Swedish-part passage rule",
    ),
    (
        "SI-NS-2027",
        "si_ns_2027",
        "Used",
        "x",
        "Program omrežja / Network Statement 2027",
        "SŽ-Infrastruktura",
        2026,
        2027,
        "EUR",
        "network_statement",
        "https://infrastruktura.sz.si/en/partners/access-to-infrastructure-for-rus/network-statement/",
        REGISTER_REVIEWED,
        "§5.3: C_P1 base with the PP/PD/PM/PV/Pl factor chain, R4 line category. No ETCS discount claimed",
    ),
    (
        "SK-ZSR-A52B",
        "sk_zsr_a52b",
        "Used",
        "x",
        "Network Statement 2027 Annex 5.2.B (Measure 2/2018)",
        "ŽSR",
        2026,
        2019,
        "EUR",
        "network_statement",
        "https://www.zsr.sk/en/railway-undertaking/infrastructure/network-statement/network-statement-2027/",
        REGISTER_REVIEWED,
        "U1+U2 train-km terms and the U3 gross-tonne-km term, track category 1. Rates unchanged since 2019 under Measure 2/2018 — hence the 2019 price basis. U3 is published per 1000 gtkm",
    ),
    (
        "UK-NR-CP7",
        "uk_nr_cp7",
        "Used",
        "x",
        "CP7 Track Usage Price List",
        "Network Rail",
        2024,
        2024,
        "GBP",
        "tariff_list",
        "https://www.networkrail.co.uk/industry-and-commercial/information-for-operators/network-statement/",
        REGISTER_REVIEWED,
        "Default Passenger Variable Usage Charge, pence per vehicle-mile, converted per train-km for a loco plus ten coaches. CP7 uses 2023/24 prices",
    ),
    (
        "CT-GETLINK-2026",
        "ct_getlink_2026",
        "Used",
        "x",
        "Fixed Link Usage Annual Statement 2026, Annexe 4",
        "Getlink (Eurotunnel)",
        2026,
        2020,
        "EUR",
        "tariff_list",
        "https://www.getlinkgroup.com/en/our-group/eurotunnel/railway-network/",
        REGISTER_REVIEWED,
        "Offer 1 regular weekly paths. Night passenger trains run at 120 km/h in off-peak periods by definition, so the off-peak row is the night-train tariff. Billed half EUR half GBP; 2020 price basis with RPI/IPC indexation still to be applied (open action 4)",
    ),
    (
        "IRG-TAC-2025",
        "irg_tac_2025",
        "Used",
        "x",
        "IRG-Rail Track Access Charges survey 2025 (updated)",
        "IRG-Rail",
        2026,
        2025,
        "EUR",
        "survey",
        "https://irg-rail.eu/irg/documents/track-access-charges-summary",
        REGISTER_REVIEWED,
        "Per-country sheets. The cross-check baseline for every calibrated country, and the primary source for DK, whose executive order is not on disk",
    ),
]

## Cross-check, conversion and method sources

The sources that do not price a specific country: the FX reference, the evidence
behind the escalation rate, and the structural references that decide what belongs
in the minimum access package at all.

In [ ]:
# --- Cross-check, FX and escalation sources ---
#
# The per-country documents above give a charge in a currency at a price
# basis. Two further conversions stand between that and a number the cost
# model can use, and each needs its own provenance: the FX rate, and the
# escalation from the document's price basis to the evaluation year.
register_rows += [
    (
        "IRG-MM-14",
        "irg_mm_14",
        "Used",
        "-",
        "IRG-Rail 14th Annual Market Monitoring Report (main report and dataset)",
        "IRG-Rail",
        2026,
        2024,
        "EUR",
        "survey",
        "https://irg-rail.eu/irg/documents/market-monitoring",
        REGISTER_REVIEWED,
        "Primary evidence for the escalation rate: passenger MAP charges per "
        "train-km rose continuously over 2020-2024 at an average 3% a year. "
        "Also the plausibility check on absolute charge levels",
    ),
    (
        "IRG-MM-9",
        "irg_mm_9",
        "Used",
        "-",
        "IRG-Rail 9th Annual Market Monitoring Report",
        "IRG-Rail",
        2021,
        2019,
        "EUR",
        "survey",
        "https://irg-rail.eu/irg/documents/market-monitoring",
        REGISTER_REVIEWED,
        "The earlier leg of the same series: passenger TAC per train-km "
        "EUR 4.13 (2015) to EUR 4.63 (2019), 2.9% a year. Two independent "
        "five-year windows agreeing on ~3% is what makes the escalation rate "
        "a trend rather than an artefact of the 2022-23 inflation spike",
    ),
    (
        "IRG-CHARGING-2020",
        "irg_charging_2020",
        "Used",
        "-",
        "Overview of Charging Practices for the Minimum Access Package in Europe",
        "IRG-Rail",
        2020,
        None,
        "EUR",
        "survey",
        "https://irg-rail.eu/irg/documents/track-access-charges-summary",
        REGISTER_REVIEWED,
        "Structural reference for how MAP charges are built per country — the "
        "basis for deciding which national components belong in the MAP scope "
        "and which are excluded as energy, station or facility charges",
    ),
    (
        "ECB-FX",
        "ecb_fx",
        "Used",
        "-",
        "ECB euro foreign exchange reference rates",
        "European Central Bank",
        2026,
        2026,
        "EUR",
        "fx_reference",
        "https://www.ecb.europa.eu/stats/policy_and_exchange_rates/euro_reference_exchange_rates/html/index.en.html",
        REGISTER_REVIEWED,
        "Reference rates behind FX_TO_EUR. Nine of the calibrated countries "
        "publish in a non-euro currency, so the FX snapshot is a calibration "
        "input in its own right and is versioned with the rest. Daily rates; "
        "the snapshot date is recorded with the table, and a scenario may "
        "legitimately pin a different one",
    ),
    (
        "ECB-PROJECTIONS",
        "ecb_projections",
        "Used",
        "-",
        "Eurosystem staff macroeconomic projections for the euro area",
        "European Central Bank",
        2025,
        None,
        "EUR",
        "macro_projection",
        "https://www.ecb.europa.eu/press/projections/html/index.en.html",
        REGISTER_REVIEWED,
        "The HICP path that the escalation rate is judged against: TAC rising "
        "~3% a year against ~2% general inflation is a real-terms increase of "
        "about a point a year, which is the claim the escalation rate makes. "
        "Also the anchor the compositions calibration uses, so both domains "
        "reach 2032 on a comparable basis",
    ),
    (
        "ERegG-37",
        "eregg_37",
        "Used",
        "-",
        "Eisenbahnregulierungsgesetz §37 and ECJ ruling C-770/24 on the Trassenpreisbremse",
        "Bundestag / Court of Justice of the European Union",
        2026,
        None,
        "EUR",
        "regulation",
        "https://curia.europa.eu/juris/liste.jsf?num=C-770/24",
        REGISTER_REVIEWED,
        "Why the German escalation is NOT extrapolated. German SPFV charges "
        "rose 19.9% over 2020-2024, far above the European average, because "
        "§37(2) capped regional increases and pushed the shortfall onto "
        "long-distance and freight. The ECJ struck that provision down in "
        "March 2026, so the mechanism behind the outlier is being unwound "
        "rather than compounding",
    ),
]

## Write and validate

In [ ]:
# --- Write and validate ---


def write_data(name: str, columns: list[str], rows: list[dict]) -> None:
    """STDLIB-ONLY writer, shared by both calib notebooks."""
    path = DATA_DIR / name
    with open(path, "w", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=columns, extrasaction="ignore")
        writer.writeheader()
        for row in rows:
            writer.writerow(
                {c: ("" if row.get(c) is None else row[c]) for c in columns}
            )
    print(f"  {name}: {len(rows)} rows")


register_dicts = [dict(zip(REGISTER_COLUMNS, r)) for r in register_rows]

ids = [r["source_id"] for r in register_dicts]
assert len(ids) == len(set(ids)), (
    f"duplicate source_id: {sorted({i for i in ids if ids.count(i) > 1})}"
)

# Every row must carry a locator a reader can follow — either a public URL
# or an explicit MISSING. Silence is the failure mode this guards against.
_blank = [r["source_id"] for r in register_dicts if not r["url_or_file"]]
assert not _blank, f"rows with no url_or_file: {_blank}"

# A gathered document is one we can open; the file name is derivable from
# the source_id, so "downloaded" is the only flag needed.
_gathered = sum(1 for r in register_dicts if r["downloaded"] == "x")

write_data("sources_register.csv", REGISTER_COLUMNS, register_dicts)
print(
    f"register: {len(register_dicts)} sources, "
    f"{sum(1 for r in register_dicts if r['used'] == 'Used')} in use, "
    f"{_gathered} documents on disk"
)